In [1]:
import pandas as pd
import os
from Bio import SeqIO, AlignIO
import pysam
# from Bio.Align.Applications import PrankCommandline
# import subprocess
from collections import Counter
import statistics
import re
import csv
from Bio.SeqIO import FastaIO
import stringdist

from multiprocessing import Pool, cpu_count
from tqdm import tqdm

import Levenshtein
import numpy as np


In [22]:
"""
call mutations
"""
def ccSeq(sub, frequency_threshold, quality_threshold):
    """
    find cluster consensus mutations in each uc group
    """
    num = len(sub)
    ccMuts = []
    
    muts =sub["mut_type"].to_numpy()
    muts = removeBracket(muts, qual = False)
    mut_quals =sub["mut_qual"].to_numpy()
    mut_quals = removeBracket(mut_quals, qual = True)
    indels =sub["indel_type"].to_numpy()
    indels = removeBracket(indels, qual = False)
    indel_quals =sub["indel_qual"].to_numpy()
    indel_quals = removeBracket(indel_quals, qual = True)
      
    fM = filterMuts(muts, mut_quals, num, frequency_threshold, quality_threshold)
    fID = filterMuts(indels, indel_quals, num, frequency_threshold, quality_threshold)
    ccMuts.append(fM)
    ccMuts.append(fID)
    ccMuts = flatList(ccMuts)
    return(ccMuts)

def removeBracket(s,qual=True):
    """
    ori: ["['2048TC']" '[]' "['1283CT', '1529GC']"]
    transformed: ['2048TC','1283CT','1529GC']
    """
    su = []
    if s is not None:
        for s_sub in s:
            s_sub = s_sub.replace("[","")
            s_sub = s_sub.replace("]","")
            s_sub = s_sub.replace("'","")
            s_sub = s_sub.split(",")
            for s_sub_sub in s_sub:
                if s_sub_sub != '':
                    s_sub_sub = s_sub_sub.replace(" ","")
                    if qual:
                        su.append(int(s_sub_sub))
                    else:
                        su.append(str(s_sub_sub))
    return(su)
    
def filterMuts(muts, mut_quals, num, frequency_threshold = 0.5, quality_threshold = 50):
    """
    filter mutations/indels with frequency >= frequency_threshold
    """
    cs = [ele for ele, cnt in Counter(muts).items() if cnt >= num*frequency_threshold]
    csMuts = []
    if len(cs) >0:
        for mutType in cs:
            quals = []
            for b,q in zip(muts, mut_quals):
                if b == mutType:
                    quals.append(q)
            if statistics.mean(quals) >= quality_threshold:
                csMuts.append(mutType)
    return(csMuts)


def flatList(Alist):
    flat_list = []
    for sublist in Alist:
        for item in sublist:
            flat_list.append(item)        
    return(flat_list)


##-----------------------------------------------------------------------
"""
create allele matrix
"""
def parseDel(D,maxLen=200):
    for s in D:
        pos = re.findall("(\d+)",s)
        if len(pos) ==2:
            if int(pos[1])-int(pos[0]) > maxLen:
                return("LongDels")
def parseSubs(subs, ref):
    pos=[re.findall(r'[0-9]+',s)[0] for s in subs]
    types=[re.findall(r'[A-Z]+',s)[0] for s in subs]
    pos=[re.findall(r'[0-9]+',s)[0] for s in subs]
    types=[re.findall(r'[A-Z]+',s)[0] for s in subs]
    pos_m=[]
    for i in pos:
        pos_m.append(int(i))
    type_m=[]
    for i in types:
        type_m.append(i)
    s1=[0]*476
    s2=ref
    for i in range(len(pos_m)):
        pointer = pos_m[i] #-1
        s1[pointer]=1
        s2 = s2[:pointer] + type_m[i][1] + s2[pointer + 1:]
    s0=''.join(map(str,s1))
    return(s0, s2)   ## s0: binary; s2:character sequence

def createMatrix(infile, OUTDIR, my_sample, REF):
    ref = open(REF,"r").readlines()[1].replace("\n", "")
    out_fasta=OUTDIR+my_sample+".fa"           ## output 1. fasta file
    out_allele=OUTDIR+my_sample+"_allele-table.txt"               ## output 2. binary file
    out_phy=OUTDIR+my_sample+"_temp.phy"                   ## output .phy file for iqtree reconstruction

    with open(infile,"r") as f_in, open(out_allele, "w") as f_allele:#, open(out_phy, "w") as f_phy, open (out_fasta, "w") as f_fasta
        f_in_lines = f_in.readlines()[1:]
        numCell = len(f_in_lines)+1
        #f_phy.write(str("ref")+" "+''.join(map(str,[0]*476))+"\n")
        #f_fasta.write(">"+str("ref")+"\n"+ref+"\n")
        for line in f_in_lines:
            muts=line.replace("\n","").split("\t")
            muts
            cell_name=muts[0]
            staticBarcode_name=muts[1]
            numUMI=muts[2]
            readCount=muts[3]
            sample=muts[6]
            muts.pop(0)
            muts.pop(0)
            muts.pop(0)
            muts.pop(0)
            muts.pop(0)
            muts = muts[0].split(",")
            types=[re.findall(r'[A-Z]+',s) for s in muts]
            l = len(types)
            subs = []
            #### deletions ############
            D = []
            for i in range(l):
                if types[i]==["D"]:
                    D.append(muts[i])
            D_long = parseDel(D,maxLen=450)
            #### insertions ###########
            I = []
            for i in range(l):
                if types[i]==["I"]:
                    I.append(muts[i])

            ### substitutions ########

            if D_long is None or "LongDels" not in D_long:
                subs = [ele for ele in muts if ele not in D and ele not in I]
                if len(subs)==1 and subs[0]=="":
                    matrix_binary = ''.join(map(str,[0]*476))
                    matrix_character = ref
                else:    
                    matrixALL = parseSubs(subs, ref)
                    matrix_binary = matrixALL[0]
                    matrix_character = matrixALL[1].replace("\n", "")

                f_allele.write(cell_name+"\t"+staticBarcode_name+"\t"+sample+"\t"+numUMI+"\t"+readCount+"\t"+matrix_character+"\t"+matrix_binary+"\n") ## add number of UMIs in each static barcode!!
                #f_phy.write(cell_name+"_"+staticBarcode_name+" "+matrix_binary+"\n")
                #f_fasta.write(">"+cell_name+"_"+staticBarcode_name+"\n"+matrix_character+"\n")
def makeConsensus(group):
    cellBC,sub = group
    
    counts=sub.groupby(['intBC','umi']).size().reset_index(name='counts')
    counts=counts[counts['counts']>=numRead]
    counts_umi=counts.groupby(['intBC']).size().reset_index(name='counts')
    counts_umi=counts_umi[counts_umi['counts']>=numUMI]

    intBC_df=pd.DataFrame()
    for my_intBC in counts_umi.intBC.unique():
        sub_df=sub[(sub['intBC']==my_intBC) & (sub['umi'].isin(counts['umi']))]
        
        ## collapse each umi group as a consensus
        umis=sub_df.umi.unique()
        intBCMuts = []
        for my_umi in umis:
            sub3=sub_df.loc[sub_df["umi"]==my_umi]
            ssMuts = ccSeq(sub3, mutFreq, quality_threshold)
            intBCMuts.append(ssMuts)
                
        ## make each static barcode as consensus!!!
        Mutation_list = Counter(flatList(intBCMuts))
        csMutation=[]
        for mut, freq in Mutation_list.items():
            if freq >= len(umis)*mutFreq:
                csMutation.append(mut)
        ## make each static barcode as consensus end
        s=len(csMutation)   ## s: number of mutations
        if s >0:
            df = pd.DataFrame([{'cellBC': cellBC[0], 'intBC': my_intBC, 'umiCount': len(umis),'readCount': sub_df.shape[0],
                                'numMut': s,'mut':','.join([i for i in csMutation])}])        
        elif s==0:
            df = pd.DataFrame([{'cellBC': cellBC[0], 'intBC': my_intBC, 'umiCount': len(umis),'readCount': sub_df.shape[0],
                                'numMut': s,'mut':''}])
        intBC_df=pd.concat([intBC_df, df], ignore_index=True)
    return(intBC_df)            
def reverse_complement(sequence):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement[base] for base in reversed(sequence))


##########################################################################################################
## error correct whitelist, at largest 3 distances
def error_correct_intBC(
    umi_table: pd.DataFrame,
    dist_thresh:int=3,
) -> pd.DataFrame:
    
    whitelist="/syn2/zhaolian/3.JiLab/results/1.BarcodeSeq/Barcode_list_filtered.txt"
    if isinstance(whitelist, list):
        whitelist_set = set(whitelist)
    else:
        with open(whitelist, "r") as f:
            whitelist_set = set(
                line.strip() for line in f if not line.isspace()
            )
    whitelist = list(whitelist_set)
    
    unique_intbcs = list(umi_table["intBC"].unique())
    corrections = {intbc: intbc for intbc in whitelist_set}
    print(f"{len(unique_intbcs)} intBCs detected.")
    intbc_dist_thresh=dist_thresh
    for intbc in tqdm(unique_intbcs, desc="Correcting intBCs to whitelist"):
        min_distance = np.inf
        min_wls = []
        if intbc not in whitelist_set:
            for wl_intbc in whitelist:
                distance = Levenshtein.distance(intbc, wl_intbc)
                if distance < min_distance:
                    min_distance = distance
                    min_wls = [wl_intbc]
                elif distance == min_distance:
                    min_wls.append(wl_intbc)
    
        # Correct only if there is one matching whitelist. Discard if there
        # are multiple possible corrections.
        if len(min_wls) == 1 and min_distance <= intbc_dist_thresh:
            corrections[intbc] = min_wls[0]
        else:
            corrections[intbc] = intbc
    
    umi_table["intBC"] = umi_table["intBC"].map(corrections)
    umi_table['inwhitelist']="No"
    umi_table.loc[umi_table['intBC'].isin(whitelist_set), 'inwhitelist'] = 'Yes'
    print(Counter(umi_table['inwhitelist']))
    unique_intbcs = list(umi_table["intBC"].unique())
    print(f"{len(unique_intbcs)} intBCs after correction.")
    return umi_table
    # MCorrected=pd.concat([MCorrected,umi_table],ignore_index=True)
    ##########################################################################################################


In [ ]:
files=["C4007-C4011","C4026-C4033","Q192-C4007","Q25-Q186-Q192"]
OUTDIR="/syn2/zhaolian/3.JiLab/results/3.PacBio/2.alleleTable/"
M=pd.DataFrame()
for my_file in files:
    df=pd.read_csv(OUTDIR+my_file+"_assigned_reads.txt",delimiter="\t")
    M=pd.concat([M, df],ignore_index=True)

M=error_correct_intBC(M,dist_thresh=3)


In [15]:
# M=MCorrected

numRead=1               ## minimum number of reads in each UMI
numUMI=1                 ## minimum number of UMIs in each static barcode
mutFreq=0.6              ## export the mutation if mutation frequency >= mutFreq 
quality_threshold = 10   ## mapping quality
REF="/syn2/zhaolian/3.JiLab/reference/PacBio_Targetonly_476bp.fasta"  #fixed 12/13/2024, before is TargetOnly476.fasta"
###########################################
for my_sample in M['sampleID'].unique():
    outfile=OUTDIR+my_sample+"_mutations_"+str(numRead)+"_"+str(numUMI)+".txt" ## final output file 2
    df=M[M['sampleID']==my_sample]
    ###########################################
    groups = [(cellBC, sub_df) for (cellBC), sub_df in df.groupby(['cellBC'])]
    # print("Done group")
    with Pool(processes=60) as pool:  #cpu_count()  
        results = list(tqdm(pool.imap(makeConsensus, groups),total=len(groups)))
        
    # Filter out None values and create DataFrame
    new = pd.concat(results)
    new['sampleID']=my_sample
    new.to_csv(outfile, sep='\t',index=False,header=True)    
    """
    """
    createMatrix(outfile, OUTDIR, my_sample,REF)


100%|██████████| 218/218 [00:00<00:00, 594.56it/s]


In [16]:
M.iloc[:3]

,mouse,sample,i7_i5,mouse_sample,qname,cellBC,umi,intBC,mut_type,mut_qual,indel_type,indel_qual,sampleID,inwhitelist
0,C4007,LLT-1,ACAGTAACTA_GATGAAGATA,C4007_LLT-1,@m84130_240730_180513_s2/143396956/ccs,GGTTAACGTACGCTTA,TAGCTGAGTTG,CATGAGTCTGTCCACCCCTTCCTCTGT,"['27CT', '29CT', '33GA', '40GA', '123GA', '155...","[40, 40, 40, 40, 40, 40, 40]","['20I', '202I', '22-24D', '203-205D', '242D']","[40, 40, 40, 40, 40]",C4007_LLT-1,Yes
1,C4007,LLT-1,ACAGTAACTA_GATGAAGATA,C4007_LLT-1,@m84130_240730_180513_s2/127205943/ccs,GGTTAACGTACGCTTA,ATAGCTGAGTTG,CATCCGTCTTGAGACCGACTCCGACGT,"['27CT', '29CT', '33GA', '40GA', '86CT', '123G...","[40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 4...","['20I', '202I', '22-24D', '203-205D']","[40, 40, 40, 40]",C4007_LLT-1,Yes
2,C4007,LLT-1,ACAGTAACTA_GATGAAGATA,C4007_LLT-1,@m84130_240730_180513_s2/191959397/ccs,GGTTAACGTACGCTTA,ATAGCTGAGTTG,CATCCGTCTTGAGACCGACTCCGACGT,"['27CT', '29CT', '33GA', '40GA', '86CT', '123G...","[40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 4...","['20I', '202I', '22-24D', '203-205D']","[40, 40, 40, 40]",C4007_LLT-1,Yes


In [23]:
# M=MCorrected

numRead=1               ## minimum number of reads in each UMI
numUMI=1                 ## minimum number of UMIs in each static barcode
mutFreq=0.6              ## export the mutation if mutation frequency >= mutFreq 
quality_threshold = 10   ## mapping quality
REF="/syn2/zhaolian/3.JiLab/reference/PacBio_Targetonly_476bp.fasta"  #fixed 12/13/2024, before is TargetOnly476.fasta"
###########################################
M=pd.read_csv('/syn2/zhaolian/3.JiLab/results/3.PacBio/2.alleleTable/name',header=None)

for sample in M[0].tolist():
    my_sample='_'.join(sample.split('_')[:2])
    outfile=OUTDIR+my_sample+"_mutations_"+str(numRead)+"_"+str(numUMI)+".txt" ## final output file 2
    new=pd.read_csv(outfile, sep='\t')
    """
    """
    createMatrix(outfile, OUTDIR, my_sample,REF)


In [19]:
OUTDIR

'/syn2/zhaolian/3.JiLab/results/3.PacBio/2.alleleTable/'